# NPS Open Climate Data — Pipeline

Run from the repo root. Exports DAYMET + ERA5 daily climate data for all 63 US
National Parks via Earth Engine batch tasks (serverless, async), aggregates the
results, and copies the JSON summaries into the Astro site.

Prerequisite: a [Google Cloud project with the Earth Engine API enabled](https://developers.google.com/earth-engine/guides/access).
A full run takes ~2–4 hours of GEE wall-clock time. Tasks run on Google's
servers, so you can close the notebook once they're submitted and come back to
step 5 once they show **COMPLETED**.


## 1. Install dependencies


In [ ]:
import os, subprocess, sys

os.chdir(os.path.dirname(os.path.abspath('pipeline.ipynb')))
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '.',
     'numpy', 'pyarrow', 'requests', 'google-api-python-client', '--quiet'],
    check=True,
)
print('Done.')


## 2. Authenticate with Earth Engine

First run only — opens a browser to authorize. `auth_mode='localhost'` captures
the token automatically (the default `'notebook'` mode shows an input box that
doesn't render in VS Code).


In [ ]:
import ee

GCP_PROJECT = 'ee-annieresearch'

ee.Authenticate(auth_mode='localhost')
ee.Initialize(project=GCP_PROJECT)
print('Earth Engine initialized.')


## 3. Submit export tasks for all parks

Submits one EE task per park (sub-units submit one each for multipart parks like
Saguaro and Channel Islands). Returns immediately. Monitor task progress at
<https://code.earthengine.google.com/tasks>.

Boundaries come from EE's `USGS/GAP/PAD-US/v20` catalog where available, with a
fallback to the committed PAD-US 4.1 GeoJSONs under `site/public/data/boundaries/`
for parks the EE asset doesn't know about — so a single run covers all 63 parks.

Set `SLUGS` to a list (e.g. `['yellowstone', 'glacier']`) to export a subset; leave
as `None` for the full batch.


In [ ]:
from nps_climate_data.batch import submit_all_tasks

DRIVE_FOLDER = 'NPS_Climate_Data'
SLUGS = None

task_infos = submit_all_tasks(
    drive_folder=DRIVE_FOLDER,
    start='1980-01-01',
    slugs=SLUGS,
)
print(f'Submitted {len(task_infos)} task(s).')


## 4. (Optional) Wait for tasks to finish

Polls every 30 seconds until every task shows COMPLETED. Skip if you'd rather
close the notebook and check the EE task page manually.


In [ ]:
from nps_climate_data.batch import wait_for_tasks

wait_for_tasks(task_infos)


## 5. Download CSVs from Google Drive

**One-time setup** — run in a terminal once per machine:
```
gcloud auth application-default login \
    --scopes=https://www.googleapis.com/auth/drive.readonly,\
             https://www.googleapis.com/auth/cloud-platform
```

Then run the cell below. CSVs land in `data/raw/<slug>/<slug>.csv`.


In [ ]:
from nps_climate_data.batch import download_from_drive

download_from_drive(
    drive_folder=DRIVE_FOLDER,
    out_root='data',
    stems=[info['stem'] for info in task_infos],
    quota_project=GCP_PROJECT,
)


## 6. Build site JSON summaries

Aggregates the daily series into annual / seasonal summaries, runs Mann–Kendall
trend tests + Theil–Sen slopes, and writes one JSON per park to `data/site/`.


In [ ]:
import subprocess

subprocess.run(['python', 'scripts/02_build_site_data.py'], check=True)
print('Site JSON written to data/site/')


## 7. Copy data into the site

Copies the JSON summaries into `site/public/data/` and gzip-compresses the raw
CSVs for the per-park download links. After this, run
`cd site && npm install && npm run dev` to preview locally.


In [ ]:
import gzip, shutil
from pathlib import Path

pub = Path('site/public/data')
(pub / 'parks').mkdir(parents=True, exist_ok=True)

shutil.copy('data/site/parks.json', pub / 'parks.json')
for f in Path('data/site/parks').glob('*.json'):
    shutil.copy(f, pub / 'parks' / f.name)

raw_dst = pub / 'raw'
raw_dst.mkdir(exist_ok=True)
for slug_dir in Path('data/raw').iterdir():
    if not slug_dir.is_dir():
        continue
    out_dir = raw_dst / slug_dir.name
    out_dir.mkdir(exist_ok=True)
    for csv in slug_dir.glob('*.csv'):
        with open(csv, 'rb') as f_in, gzip.open(out_dir / (csv.name + '.gz'), 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

n_json = len(list((pub / 'parks').glob('*.json')))
n_csv = len(list(raw_dst.rglob('*.gz')))
print(f'Copied {n_json} park JSON files and {n_csv} gzipped CSVs.')
